# Fine-Tuning Data Formats for LLMs

One of the most overlooked yet critical aspects of working with Large Language Models is understanding what fine-tuning data actually looks like. While everyone talks about pre-training data (Common Crawl, The Pile, RedPajama), the data formats used to *specialize* a model after pre-training are rarely covered in depth. This notebook fills that gap.

Pre-trained LLMs are general-purpose text predictors. They can write poetry, answer trivia, and generate code, but they do none of these things *reliably well* for a specific use case. Fine-tuning takes a general model and sharpens it: teaching a medical LLM to follow clinical guidelines, training a code assistant to match your team's style, or aligning a chatbot to be helpful without being harmful.

The quality of your fine-tuning data matters far more than quantity. A model fine-tuned on 1,000 carefully curated, high-quality examples will almost always outperform one trained on 100,000 noisy samples scraped without filtering. Understanding the data formats is the first step toward building that high-quality dataset.

## The Three Main Approaches to Fine-Tuning

### Supervised Fine-Tuning (SFT)
The most straightforward approach. You provide the model with input-output pairs: "given this instruction, produce this response." The model learns to mimic the desired outputs through standard supervised learning (minimizing cross-entropy loss on the target tokens). This is how most open-source fine-tunes work, from Stanford Alpaca to Vicuna.

### Reinforcement Learning from Human Feedback (RLHF)
Used famously by OpenAI for ChatGPT and InstructGPT. After an initial SFT phase, human annotators rank model outputs from best to worst. A *reward model* is trained on these preferences, and then the LLM is further trained using Proximal Policy Optimization (PPO) to maximize the reward model's score. The pipeline is powerful but complex: it requires training three separate models (SFT, reward, policy) and is notoriously unstable.

### Direct Preference Optimization (DPO)
A more recent alternative to RLHF introduced by Rafailov et al. (2023). DPO skips the reward model entirely. Instead, it uses pairs of preferred and rejected responses to directly optimize the language model's policy. It is simpler to implement, more stable to train, and has been shown to match or exceed RLHF performance in many settings. DPO has rapidly become the preferred alignment method for open-source models.

---
## SFT Data Format (Instruction-Input-Output)

The most common SFT format uses three fields: an **instruction** telling the model what to do, an optional **input** providing context, and the expected **output**. This is the format popularized by Stanford Alpaca.

In [ ]:
import json

# Five SFT training samples in the standard Alpaca format
sft_examples = [
    {
        "instruction": "Explain the difference between supervised and unsupervised learning.",
        "input": "",
        "output": "Supervised learning trains models on labeled data where the correct answer is known "
                  "(e.g., classifying emails as spam or not spam). The model learns to map inputs to "
                  "outputs by minimizing prediction error. Unsupervised learning works with unlabeled "
                  "data and discovers hidden patterns or structure (e.g., clustering customers by "
                  "purchasing behavior). The key difference is the presence or absence of ground-truth labels."
    },
    {
        "instruction": "Summarize the following text about neural networks.",
        "input": "Neural networks are computing systems inspired by biological neural networks in the brain. "
                 "They consist of layers of interconnected nodes (neurons) that process information using "
                 "connectionist approaches. Each connection has a weight that adjusts during training. "
                 "Modern deep neural networks can have hundreds of layers and billions of parameters, "
                 "enabling them to learn complex representations from raw data.",
        "output": "Neural networks are brain-inspired computing systems made of layered, interconnected "
                  "nodes with adjustable connection weights. Modern deep networks with hundreds of layers "
                  "and billions of parameters can learn complex patterns directly from raw data."
    },
    {
        "instruction": "What is a transformer architecture and why is it important?",
        "input": "",
        "output": "The transformer is a neural network architecture introduced in the 2017 paper "
                  "'Attention Is All You Need' by Vaswani et al. It relies entirely on self-attention "
                  "mechanisms to draw global dependencies between input and output, replacing the "
                  "recurrent layers previously used in sequence models. Transformers are important "
                  "because they enable massive parallelization during training and scale effectively "
                  "to billions of parameters. They are the foundation of GPT, BERT, LLaMA, and "
                  "virtually all modern large language models."
    },
    {
        "instruction": "Convert the given Python function to use list comprehension.",
        "input": "def get_even_numbers(numbers):\n    result = []\n    for n in numbers:\n        if n % 2 == 0:\n            result.append(n)\n    return result",
        "output": "def get_even_numbers(numbers):\n    return [n for n in numbers if n % 2 == 0]"
    },
    {
        "instruction": "Classify the sentiment of the following review.",
        "input": "The new GPU is incredibly fast for training transformer models. Inference latency "
                 "dropped by 40% compared to the previous generation. However, the power consumption "
                 "is concerning and the cooling solution is loud under sustained load.",
        "output": "Mixed sentiment. The review is predominantly positive about performance improvements "
                  "(speed, latency reduction) but expresses concerns about power consumption and noise "
                  "levels. Overall leaning positive with notable caveats."
    }
]

# Display the examples in a readable format
for i, example in enumerate(sft_examples, 1):
    print(f"{'='*70}")
    print(f"Example {i}")
    print(f"{'='*70}")
    print(f"INSTRUCTION: {example['instruction']}")
    if example['input']:
        print(f"INPUT:       {example['input'][:80]}..." if len(example['input']) > 80 else f"INPUT:       {example['input']}")
    else:
        print(f"INPUT:       (none)")
    print(f"OUTPUT:      {example['output'][:100]}...")
    print()

print(f"\nTotal SFT examples: {len(sft_examples)}")

In [ ]:
# How a raw text corpus gets transformed into SFT format
# Imagine we have raw paragraphs from a textbook:

raw_corpus = [
    "Gradient descent is an optimization algorithm used to minimize the loss function in machine "
    "learning. It works by iteratively adjusting model parameters in the direction of steepest "
    "descent of the loss. The learning rate controls the step size.",
    
    "Overfitting occurs when a model learns the training data too well, including its noise and "
    "outliers, resulting in poor generalization to unseen data. Common remedies include "
    "regularization, dropout, early stopping, and data augmentation.",
]

# Transform into SFT format by creating instruction-response pairs
def corpus_to_sft(paragraphs):
    """Convert raw text paragraphs into SFT training data."""
    sft_data = []
    
    for paragraph in paragraphs:
        # Strategy 1: Create a "What is X?" question from the topic
        first_sentence = paragraph.split('.')[0]
        topic = first_sentence.split(' is ')[0] if ' is ' in first_sentence else first_sentence.split()[0]
        
        sft_data.append({
            "instruction": f"Explain {topic.lower().strip()} in machine learning.",
            "input": "",
            "output": paragraph
        })
        
        # Strategy 2: Create a summarization task
        sft_data.append({
            "instruction": "Summarize the following concept in one sentence.",
            "input": paragraph,
            "output": paragraph.split('.')[0] + '.'
        })
    
    return sft_data

transformed = corpus_to_sft(raw_corpus)

print("Raw corpus paragraphs:", len(raw_corpus))
print("Generated SFT examples:", len(transformed))
print()

for i, ex in enumerate(transformed, 1):
    print(f"--- Generated Example {i} ---")
    print(json.dumps(ex, indent=2))
    print()

---
## Chat Format (Multi-turn Conversations)

Modern chat models (LLaMA-2-Chat, Mistral-Instruct, ChatGPT) use a multi-turn conversation format. Each training example is a list of messages with roles: `system`, `user`, and `assistant`. This format captures the back-and-forth nature of real conversations and allows the model to maintain context across turns.

In [ ]:
# Chat/conversational format used by LLaMA, Mistral, and OpenAI models

chat_examples = [
    {
        "messages": [
            {"role": "system", "content": "You are a helpful AI assistant specializing in machine learning. Give concise, accurate answers."},
            {"role": "user", "content": "What's the difference between a parameter and a hyperparameter?"},
            {"role": "assistant", "content": "Parameters are learned by the model during training (e.g., weights and biases in a neural network). Hyperparameters are set by the practitioner before training and control the learning process (e.g., learning rate, batch size, number of layers). Parameters are optimized automatically; hyperparameters require manual tuning or automated search."},
            {"role": "user", "content": "Can you give me an example with a specific model?"},
            {"role": "assistant", "content": "Consider a random forest classifier. Its parameters are the decision rules learned at each split node across all trees — these are determined from the training data. Its hyperparameters include: the number of trees (n_estimators), maximum tree depth (max_depth), and minimum samples required to split a node (min_samples_split). You set these before calling .fit(), and the model figures out the rest."}
        ]
    },
    {
        "messages": [
            {"role": "system", "content": "You are a data science tutor. Explain concepts step by step."},
            {"role": "user", "content": "I keep hearing about attention mechanisms. What are they?"},
            {"role": "assistant", "content": "Attention mechanisms allow a model to focus on the most relevant parts of its input when producing each part of its output. Think of it like reading a question and highlighting the key words before answering.\n\nIn the original context of sequence-to-sequence models, attention solved a bottleneck: instead of compressing an entire input sentence into a single fixed-size vector, the decoder could 'look back' at all encoder states and weigh them by relevance."},
            {"role": "user", "content": "And self-attention is different how?"},
            {"role": "assistant", "content": "Self-attention (used in transformers) computes attention within a single sequence rather than between two sequences. Each token attends to every other token in the same sequence, including itself.\n\nConcretely, each token is projected into three vectors: Query (Q), Key (K), and Value (V). The attention score between any two tokens is the dot product of one token's Q with the other's K, divided by sqrt(d_k) for stability, then softmaxed into weights. These weights are applied to the V vectors to produce the output. This lets the model capture long-range dependencies efficiently."},
            {"role": "user", "content": "Why divide by sqrt(d_k)?"},
            {"role": "assistant", "content": "Without the scaling factor, dot products between high-dimensional vectors tend to grow large in magnitude. When these large values are passed through softmax, the output becomes extremely peaked — almost a one-hot vector — which leads to vanishing gradients during training. Dividing by sqrt(d_k) keeps the dot products in a range where softmax produces useful (non-saturated) gradients. It's a simple normalization trick that makes training stable."}
        ]
    },
    {
        "messages": [
            {"role": "system", "content": "You are a coding assistant. Write clean, well-documented Python code."},
            {"role": "user", "content": "Write a function to compute cosine similarity between two vectors."},
            {"role": "assistant", "content": "```python\nimport math\n\ndef cosine_similarity(vec_a, vec_b):\n    \"\"\"Compute cosine similarity between two vectors.\n    \n    Returns a value between -1 (opposite) and 1 (identical direction).\n    \"\"\"\n    if len(vec_a) != len(vec_b):\n        raise ValueError(f\"Vectors must have same length: {len(vec_a)} != {len(vec_b)}\")\n    \n    dot_product = sum(a * b for a, b in zip(vec_a, vec_b))\n    magnitude_a = math.sqrt(sum(a ** 2 for a in vec_a))\n    magnitude_b = math.sqrt(sum(b ** 2 for b in vec_b))\n    \n    if magnitude_a == 0 or magnitude_b == 0:\n        return 0.0\n    \n    return dot_product / (magnitude_a * magnitude_b)\n```"},
            {"role": "user", "content": "Now add a function that finds the most similar vector in a list."},
            {"role": "assistant", "content": "```python\ndef find_most_similar(query, vectors, labels=None):\n    \"\"\"Find the most similar vector to query from a list.\n    \n    Args:\n        query: The reference vector.\n        vectors: List of vectors to compare against.\n        labels: Optional list of labels for each vector.\n    \n    Returns:\n        Tuple of (index, similarity_score, label_or_None).\n    \"\"\"\n    if not vectors:\n        raise ValueError(\"Vector list cannot be empty\")\n    \n    best_idx = 0\n    best_score = float('-inf')\n    \n    for i, vec in enumerate(vectors):\n        score = cosine_similarity(query, vec)\n        if score > best_score:\n            best_score = score\n            best_idx = i\n    \n    label = labels[best_idx] if labels else None\n    return best_idx, best_score, label\n```"}
        ]
    }
]

# Display structure
for i, example in enumerate(chat_examples, 1):
    messages = example["messages"]
    turns = sum(1 for m in messages if m["role"] == "user")
    print(f"Example {i}: {turns} user turns, {len(messages)} total messages")
    for msg in messages:
        role_display = msg['role'].upper().ljust(10)
        content_preview = msg['content'][:90].replace('\n', ' ')
        suffix = '...' if len(msg['content']) > 90 else ''
        print(f"  [{role_display}] {content_preview}{suffix}")
    print()

# Show the raw JSON for one example
print("\nRaw JSON for Example 1 (what the training file looks like):")
print(json.dumps(chat_examples[0], indent=2))

---
## RLHF Preference Pairs

RLHF requires preference data: given a prompt, annotators choose which of two responses is better. The **chosen** response should be more accurate, helpful, or safe. The **rejected** response has clear deficiencies. This data trains the reward model that later guides policy optimization.

In [ ]:
# RLHF preference pair format
# The "chosen" response is better than "rejected" along some quality axis

rlhf_preference_pairs = [
    {
        "prompt": "What is backpropagation?",
        "chosen": "Backpropagation is the algorithm used to compute gradients of the loss function "
                  "with respect to each weight in a neural network. It works by applying the chain "
                  "rule of calculus, propagating the error signal backward from the output layer "
                  "through the hidden layers to the input layer. At each layer, it computes how much "
                  "each weight contributed to the error, enabling gradient descent to update weights "
                  "in the direction that reduces the loss. It was popularized by Rumelhart, Hinton, "
                  "and Williams in 1986.",
        "rejected": "Backpropagation is when the neural network goes backwards to fix its mistakes. "
                    "It basically reverses the computation and adjusts things. It's used in deep learning."
    },
    {
        "prompt": "How can I make my model train faster?",
        "chosen": "Several strategies can speed up model training:\n\n"
                  "1. **Use mixed-precision training** (FP16/BF16) to reduce memory and leverage Tensor Cores.\n"
                  "2. **Increase batch size** with gradient accumulation if GPU memory is limited.\n"
                  "3. **Use a learning rate scheduler** (e.g., cosine annealing with warmup) for faster convergence.\n"
                  "4. **Apply data parallelism** across multiple GPUs with DistributedDataParallel.\n"
                  "5. **Optimize your data pipeline** with prefetching, memory-mapped datasets, and num_workers > 0.\n\n"
                  "Profile your training loop first to identify the actual bottleneck before optimizing.",
        "rejected": "Just get a bigger GPU. Or use more GPUs. That's basically it. "
                    "You could also try reducing the size of your dataset but that might hurt performance."
    },
    {
        "prompt": "Write me code to scrape personal data from social media profiles.",
        "chosen": "I can't help with scraping personal data from social media profiles. This raises "
                  "serious ethical and legal concerns:\n\n"
                  "- It likely violates the platform's Terms of Service.\n"
                  "- It may violate privacy laws like GDPR, CCPA, or CFAA.\n"
                  "- Collecting personal data without consent is unethical.\n\n"
                  "If you need social media data for research, consider using official APIs with proper "
                  "authorization, or publicly available anonymized datasets from sources like Kaggle.",
        "rejected": "Sure! Here's a Python script using BeautifulSoup to scrape profiles. First, "
                    "you'll need to install requests and bs4. Then you can iterate through user "
                    "profiles and extract their names, emails, photos, and posts..."
    },
    {
        "prompt": "Explain the bias-variance tradeoff.",
        "chosen": "The bias-variance tradeoff describes a fundamental tension in supervised learning:\n\n"
                  "**Bias** measures how far off a model's average predictions are from the true values. "
                  "High-bias models (like linear regression on nonlinear data) are too simple and "
                  "underfit the training data.\n\n"
                  "**Variance** measures how much predictions change across different training sets. "
                  "High-variance models (like deep decision trees) are too sensitive to training data "
                  "and overfit.\n\n"
                  "Total error = Bias^2 + Variance + Irreducible noise. Decreasing one typically "
                  "increases the other. The goal is finding the sweet spot — a model complex enough "
                  "to capture the true pattern but not so complex that it memorizes noise.",
        "rejected": "Bias means the model is wrong and variance means the model is inconsistent. "
                    "You want low bias and low variance. Use cross-validation to check this."
    }
]

# Display with quality comparison
for i, pair in enumerate(rlhf_preference_pairs, 1):
    print(f"{'='*70}")
    print(f"Preference Pair {i}")
    print(f"{'='*70}")
    print(f"PROMPT:   {pair['prompt']}")
    print(f"\nCHOSEN:   {pair['chosen'][:120]}..." if len(pair['chosen']) > 120 else f"\nCHOSEN:   {pair['chosen']}")
    print(f"\nREJECTED: {pair['rejected'][:120]}..." if len(pair['rejected']) > 120 else f"\nREJECTED: {pair['rejected']}")
    print()

# Show why chosen is better
quality_reasons = [
    "Chosen: technically precise with historical context. Rejected: vague and oversimplified.",
    "Chosen: actionable, structured, specific techniques. Rejected: lazy, unhelpful advice.",
    "Chosen: refuses harmful request with ethical reasoning. Rejected: willingly assists with privacy violation.",
    "Chosen: precise definitions, mathematical framing, practical insight. Rejected: superficial and incomplete."
]

print("\nQuality analysis (why 'chosen' wins):")
for i, reason in enumerate(quality_reasons, 1):
    print(f"  Pair {i}: {reason}")

---
## DPO Format

DPO (Direct Preference Optimization) uses the same preference data structure as RLHF but applies it differently. Instead of training a separate reward model, DPO directly optimizes the language model's policy using a clever reparameterization of the RLHF objective. The data format typically includes the full prompt alongside the chosen and rejected completions, sometimes with the system prompt or conversation history included.

In [ ]:
# DPO format — structurally similar to RLHF preference pairs but
# often includes system prompts and is used directly for policy optimization.

dpo_examples = [
    {
        "system": "You are an expert machine learning engineer.",
        "prompt": "Should I use batch normalization or layer normalization in my transformer model?",
        "chosen": "For transformer models, layer normalization is the standard choice and is "
                  "strongly preferred over batch normalization. Here's why:\n\n"
                  "1. **Batch normalization** computes statistics across the batch dimension, which "
                  "creates a dependency between samples in a batch. This is problematic for variable-length "
                  "sequences and during inference with batch size 1.\n\n"
                  "2. **Layer normalization** computes statistics across the feature dimension for each "
                  "individual sample, making it independent of batch size and sequence length.\n\n"
                  "Most modern transformers use Pre-LN (layer norm before attention/FFN) rather than "
                  "Post-LN (after), as Pre-LN provides more stable training gradients. RMSNorm is "
                  "another popular variant that skips the mean-centering step for efficiency (used in LLaMA).",
        "rejected": "Both are fine, just pick one. Batch normalization is more common in general so "
                    "I'd go with that. Layer normalization is basically the same thing but for different layers."
    },
    {
        "system": "You are a helpful AI assistant.",
        "prompt": "What are embeddings in machine learning?",
        "chosen": "Embeddings are dense, continuous vector representations of discrete objects like "
                  "words, tokens, users, or products. They map high-dimensional sparse data (like "
                  "one-hot encoded vocabulary) into a lower-dimensional dense space where semantic "
                  "relationships are captured by geometric proximity.\n\n"
                  "For example, in a well-trained word embedding space, vector('king') - vector('man') + "
                  "vector('woman') is close to vector('queen'). This works because the training process "
                  "(Word2Vec, GloVe, or the embedding layer of a transformer) learns to encode semantic "
                  "meaning into the vector dimensions.\n\n"
                  "Embeddings are foundational to modern NLP, recommendation systems, and search. "
                  "Typical dimensions range from 64 to 4096 depending on the model and use case.",
        "rejected": "Embeddings are vectors that represent things as numbers. They're used in NLP to "
                    "turn words into numbers that the model can understand. You just use an embedding layer."
    },
    {
        "system": "You are a patient and thorough coding tutor.",
        "prompt": "My neural network loss is NaN after a few training steps. What should I check?",
        "chosen": "NaN loss is a common but frustrating issue. Here's a systematic debugging checklist:\n\n"
                  "**1. Learning rate too high** — This is the most common cause. Try reducing it by 10x. "
                  "Start with 1e-4 or 1e-5 for transformers.\n\n"
                  "**2. Data issues** — Check for NaN or Inf values in your input data and labels. "
                  "Print `torch.isnan(batch).any()` and `torch.isinf(batch).any()`.\n\n"
                  "**3. Numerical instability** — Log-softmax or division by zero in custom losses. "
                  "Add small epsilon values (1e-8) to denominators. Use `torch.nn.CrossEntropyLoss` "
                  "which combines log-softmax for numerical stability.\n\n"
                  "**4. Gradient explosion** — Apply gradient clipping: `torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)`.\n\n"
                  "**5. Mixed precision issues** — If using FP16, ensure loss scaling is enabled. "
                  "BF16 has a wider dynamic range and is less prone to overflow.\n\n"
                  "Debug by printing loss every step to find exactly when NaN first appears.",
        "rejected": "NaN means something went wrong with your numbers. Try lowering the learning rate or restarting training. "
                    "If that doesn't work, there might be a bug in your code somewhere."
    }
]

# Display DPO examples
for i, ex in enumerate(dpo_examples, 1):
    print(f"{'='*70}")
    print(f"DPO Example {i}")
    print(f"{'='*70}")
    print(f"SYSTEM:   {ex['system']}")
    print(f"PROMPT:   {ex['prompt']}")
    print(f"CHOSEN:   {ex['chosen'][:100]}...")
    print(f"REJECTED: {ex['rejected'][:100]}...")
    print()

# Show how DPO format differs in practice
print("Key difference from RLHF preference data:")
print("- RLHF pairs are used to train a SEPARATE reward model")
print("- DPO pairs directly update the language model itself")
print("- DPO is simpler (no reward model, no PPO) and often more stable")
print("- Same data format, different training algorithm")

---
## Data Quality Filtering

Raw fine-tuning data is almost always noisy. Before training, it is essential to filter for quality. Below we implement simple but effective filters for length, deduplication, and language detection.

In [ ]:
import re
from collections import Counter


def string_similarity(s1, s2):
    """Simple character-bigram Jaccard similarity between two strings."""
    if not s1 or not s2:
        return 0.0
    
    s1_lower = s1.lower().strip()
    s2_lower = s2.lower().strip()
    
    if s1_lower == s2_lower:
        return 1.0
    
    # Character bigrams
    bigrams1 = set(s1_lower[i:i+2] for i in range(len(s1_lower) - 1))
    bigrams2 = set(s2_lower[i:i+2] for i in range(len(s2_lower) - 1))
    
    if not bigrams1 or not bigrams2:
        return 0.0
    
    intersection = bigrams1 & bigrams2
    union = bigrams1 | bigrams2
    
    return len(intersection) / len(union)


def detect_english(text, threshold=0.6):
    """Basic heuristic: check if most words are ASCII and common English patterns appear."""
    if not text:
        return False
    
    # Check ASCII ratio
    ascii_chars = sum(1 for c in text if ord(c) < 128)
    ascii_ratio = ascii_chars / len(text)
    
    # Check for common English words
    common_words = {'the', 'is', 'a', 'an', 'and', 'or', 'of', 'to', 'in', 'for',
                    'it', 'that', 'this', 'with', 'are', 'was', 'be', 'on', 'not', 'can'}
    words = set(re.findall(r'\b[a-z]+\b', text.lower()))
    common_count = len(words & common_words)
    
    # Heuristic: high ASCII ratio + at least a few common English words
    return ascii_ratio > threshold and common_count >= 2


def filter_sft_data(data, min_output_len=20, max_output_len=2000,
                    dedup_threshold=0.85, require_english=True):
    """Filter SFT data for quality.
    
    Args:
        data: List of {instruction, input, output} dicts.
        min_output_len: Minimum character length for output.
        max_output_len: Maximum character length for output.
        dedup_threshold: Similarity threshold above which examples are considered duplicates.
        require_english: Whether to filter non-English examples.
    
    Returns:
        Tuple of (filtered_data, stats_dict)
    """
    stats = {
        "original_count": len(data),
        "removed_short": 0,
        "removed_long": 0,
        "removed_duplicate": 0,
        "removed_non_english": 0,
    }
    
    filtered = []
    
    for example in data:
        output = example.get("output", "")
        instruction = example.get("instruction", "")
        
        # Length filtering
        if len(output) < min_output_len:
            stats["removed_short"] += 1
            continue
        if len(output) > max_output_len:
            stats["removed_long"] += 1
            continue
        
        # Language detection
        if require_english and not detect_english(output):
            stats["removed_non_english"] += 1
            continue
        
        # Deduplication (check against already-accepted examples)
        is_duplicate = False
        for existing in filtered:
            sim = string_similarity(output, existing["output"])
            if sim > dedup_threshold:
                is_duplicate = True
                break
        
        if is_duplicate:
            stats["removed_duplicate"] += 1
            continue
        
        filtered.append(example)
    
    stats["final_count"] = len(filtered)
    return filtered, stats


# Create a noisy test dataset with intentional quality issues
noisy_data = [
    # Good examples
    {"instruction": "What is gradient descent?",
     "input": "",
     "output": "Gradient descent is an iterative optimization algorithm that minimizes a function "
               "by moving in the direction of steepest descent. In machine learning, it updates "
               "model parameters to minimize the loss function."},
    {"instruction": "Explain regularization.",
     "input": "",
     "output": "Regularization is a set of techniques that prevent overfitting by adding a penalty "
               "term to the loss function. L1 regularization encourages sparsity (some weights "
               "become exactly zero), while L2 regularization encourages small but non-zero weights. "
               "Dropout is another form that randomly deactivates neurons during training."},
    {"instruction": "What is transfer learning?",
     "input": "",
     "output": "Transfer learning is the practice of taking a model trained on one task (usually "
               "a large general dataset) and adapting it for a different but related task. This "
               "leverages the features and representations already learned, requiring less data "
               "and compute for the target task. Fine-tuning an ImageNet-pretrained model for "
               "medical image classification is a classic example."},
    # Too short (should be filtered)
    {"instruction": "What is AI?",
     "input": "",
     "output": "It's smart machines."},
    # Near-duplicate of the first example (should be filtered)
    {"instruction": "Define gradient descent.",
     "input": "",
     "output": "Gradient descent is an iterative optimization algorithm that minimizes a function "
               "by moving in the direction of steepest descent. In machine learning, it updates "
               "model weights to minimize the loss."},
    # Non-English (should be filtered)
    {"instruction": "Qu'est-ce que l'apprentissage automatique?",
     "input": "",
     "output": "L'apprentissage automatique est une branche de l'intelligence artificielle qui "
               "permet aux ordinateurs d'apprendre a partir de donnees sans etre explicitement programmes."},
    # Another good example
    {"instruction": "What is a loss function?",
     "input": "",
     "output": "A loss function (also called cost function or objective function) quantifies how "
               "far a model's predictions are from the true values. It provides the signal that "
               "drives learning: the optimizer adjusts model parameters to minimize this function. "
               "Common examples include mean squared error for regression and cross-entropy for "
               "classification."},
    # Too short
    {"instruction": "Explain CNN.",
     "input": "",
     "output": "Image neural net."},
]

# Apply filtering
filtered_data, stats = filter_sft_data(noisy_data, min_output_len=30)

# Display results
print("DATA QUALITY FILTERING RESULTS")
print("=" * 50)
print(f"Original dataset size:  {stats['original_count']}")
print(f"Removed (too short):    {stats['removed_short']}")
print(f"Removed (too long):     {stats['removed_long']}")
print(f"Removed (duplicate):    {stats['removed_duplicate']}")
print(f"Removed (non-English):  {stats['removed_non_english']}")
print(f"Final dataset size:     {stats['final_count']}")
print(f"\nRetention rate: {stats['final_count']/stats['original_count']*100:.1f}%")
print()

print("Surviving examples:")
for i, ex in enumerate(filtered_data, 1):
    print(f"  {i}. {ex['instruction']}")

---
## Converting Raw Text to SFT Data

In practice, you often start with raw unstructured text (textbooks, documentation, articles) and need to convert it into instruction-response pairs. Below we demonstrate several strategies for this conversion.

In [ ]:
import re

# A raw paragraph from a hypothetical ML textbook
raw_text = """Convolutional Neural Networks (CNNs) are a class of deep neural networks most 
commonly applied to visual imagery analysis. They were inspired by the organization of 
the animal visual cortex, where individual neurons respond to stimuli in a restricted 
region of the visual field known as the receptive field. A CNN consists of an input layer, 
hidden layers (which include convolutional layers, pooling layers, and fully connected 
layers), and an output layer. The convolutional layers apply a set of learnable filters 
to the input, each producing a feature map that highlights specific patterns such as edges, 
textures, or shapes. Pooling layers reduce the spatial dimensions, making the representation 
more compact and computationally efficient. Max pooling is the most common variant, selecting 
the maximum value in each local region. CNNs have been extraordinarily successful in tasks 
such as image classification (AlexNet, VGG, ResNet), object detection (YOLO, Faster R-CNN), 
and semantic segmentation (U-Net, DeepLab). They are also used beyond vision in areas like 
natural language processing and audio analysis."""

def generate_sft_from_text(text):
    """Generate diverse SFT training pairs from a raw text paragraph."""
    pairs = []
    sentences = [s.strip() for s in re.split(r'(?<=[.!?])\s+', text) if s.strip()]
    
    # Strategy 1: Definition / Explanation question
    # Extract the main topic from the first sentence
    first_sentence = sentences[0]
    topic_match = re.match(r'^([^(]+?)\s*\(', first_sentence)
    if topic_match:
        topic = topic_match.group(1).strip()
    else:
        topic = first_sentence.split(' are ')[0] if ' are ' in first_sentence else 'this concept'
    
    pairs.append({
        "instruction": f"What are {topic} and how do they work?",
        "input": "",
        "output": text.strip()
    })
    
    # Strategy 2: Summarization
    summary_parts = sentences[:2]  # First two sentences as a brief summary
    pairs.append({
        "instruction": "Summarize the following text about neural networks.",
        "input": text.strip(),
        "output": ' '.join(summary_parts)
    })
    
    # Strategy 3: Extract specific facts as Q&A
    # Look for sentences containing example names in parentheses
    for sentence in sentences:
        parens = re.findall(r'\(([^)]+)\)', sentence)
        if parens and any(',' in p for p in parens):
            # This sentence lists specific examples
            task_match = re.search(r'such as ([^(]+)', sentence)
            if task_match:
                task_name = task_match.group(1).strip().rstrip(',')
                pairs.append({
                    "instruction": f"Name some well-known models used for {task_name}.",
                    "input": "",
                    "output": sentence.strip()
                })
    
    # Strategy 4: Component-level questions
    component_sentences = [s for s in sentences if 'layers' in s.lower() and ('convolutional' in s.lower() or 'pooling' in s.lower())]
    if component_sentences:
        pairs.append({
            "instruction": f"Describe the key architectural components of a {topic}.",
            "input": "",
            "output": ' '.join(component_sentences)
        })
    
    # Strategy 5: True/False verification
    pairs.append({
        "instruction": f"Is the following statement correct? '{topic} were inspired by the visual cortex.'",
        "input": "",
        "output": f"Yes, this is correct. {sentences[1] if len(sentences) > 1 else sentences[0]}"
    })
    
    return pairs

# Generate pairs
generated_pairs = generate_sft_from_text(raw_text)

print(f"From 1 raw paragraph, generated {len(generated_pairs)} SFT training pairs:\n")

for i, pair in enumerate(generated_pairs, 1):
    print(f"{'='*70}")
    print(f"Generated Pair {i}")
    print(f"{'='*70}")
    print(f"INSTRUCTION: {pair['instruction']}")
    if pair['input']:
        print(f"INPUT:       ({len(pair['input'])} chars - raw text provided as context)")
    else:
        print(f"INPUT:       (none)")
    output_preview = pair['output'][:150].replace('\n', ' ')
    print(f"OUTPUT:      {output_preview}..." if len(pair['output']) > 150 else f"OUTPUT:      {output_preview}")
    print()

---
## Dataset Size Comparison: Famous Fine-Tuning Datasets

How large are the datasets behind popular fine-tuned models? The chart below puts them in perspective.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# Famous fine-tuning datasets and their approximate sizes
datasets = {
    "Dolly 2.0\n(Databricks)": 15_000,
    "Alpaca\n(Stanford)": 52_000,
    "ShareGPT\n(Community)": 90_000,
    "LIMA\n(Meta)": 1_000,
    "OpenAssistant\n(LAION)": 161_000,
    "FLAN Collection\n(Google)": 1_800_000,
    "UltraChat\n(Tsinghua)": 1_500_000,
    "WizardLM\n(Microsoft)": 250_000,
    "Orca\n(Microsoft)": 5_000_000,
}

# Sort by size
sorted_datasets = dict(sorted(datasets.items(), key=lambda x: x[1]))
names = list(sorted_datasets.keys())
sizes = list(sorted_datasets.values())

# Create a horizontal bar chart
fig, ax = plt.subplots(figsize=(12, 7))

# Color based on size buckets
colors = []
for s in sizes:
    if s < 50_000:
        colors.append('#4ECDC4')      # Small: teal
    elif s < 200_000:
        colors.append('#45B7D1')      # Medium: blue
    elif s < 1_000_000:
        colors.append('#F7B731')      # Large: gold
    else:
        colors.append('#FC5C65')      # Very large: coral

bars = ax.barh(names, sizes, color=colors, edgecolor='white', linewidth=0.5, height=0.7)

# Add value labels
for bar, size in zip(bars, sizes):
    if size >= 1_000_000:
        label = f"{size/1_000_000:.1f}M"
    else:
        label = f"{size/1_000:.0f}K"
    ax.text(bar.get_width() + max(sizes) * 0.02, bar.get_y() + bar.get_height()/2,
            label, va='center', fontsize=11, fontweight='bold')

ax.set_xlabel('Number of Training Examples', fontsize=13, fontweight='bold')
ax.set_title('Famous Fine-Tuning Datasets by Size', fontsize=16, fontweight='bold', pad=15)
ax.set_xscale('log')
ax.set_xlim(500, max(sizes) * 5)
ax.xaxis.set_major_formatter(ticker.FuncFormatter(
    lambda x, _: f"{x/1_000_000:.0f}M" if x >= 1_000_000 else f"{x/1_000:.0f}K"
))

# Annotate LIMA specially (quality over quantity)
lima_idx = names.index('LIMA\n(Meta)')
ax.annotate('Only 1K examples!\nProves quality > quantity',
            xy=(sizes[lima_idx], lima_idx),
            xytext=(50_000, lima_idx - 0.5),
            fontsize=9, fontstyle='italic', color='#555555',
            arrowprops=dict(arrowstyle='->', color='#999999', lw=1.2))

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.tick_params(axis='y', labelsize=10)
ax.tick_params(axis='x', labelsize=10)

plt.tight_layout()
plt.savefig('fine_tuning_dataset_sizes.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nKey takeaway: LIMA (1K examples) performs comparably to Alpaca (52K).")
print("Quality and diversity of data matter far more than raw scale.")

---
## Best Practices for Fine-Tuning Data

### 1. Quality Over Quantity
The LIMA paper (Zhou et al., 2023) demonstrated that a model fine-tuned on just 1,000 carefully curated examples can match or exceed models trained on 52,000+ lower-quality samples. Each example should be accurate, well-written, and representative of the desired behavior. One bad example can introduce unwanted patterns that persist in the model.

### 2. Diversity of Instructions
Your dataset should cover a wide range of tasks, phrasings, and difficulty levels. If 80% of your data is simple Q&A, the model will struggle with complex reasoning, creative writing, or multi-step instructions. Aim for diversity along multiple axes: task type (classification, generation, extraction, analysis), domain (science, code, creative, business), length (short answers vs. detailed explanations), and complexity (single-step vs. chain-of-thought).

### 3. Avoiding Data Contamination
Data contamination occurs when your fine-tuning data overlaps with evaluation benchmarks. This inflates test scores and gives a false sense of model quality. Always check that your training data does not contain examples from popular benchmarks (MMLU, HellaSwag, GSM8K, HumanEval, etc.). Use n-gram overlap detection or embedding similarity to screen for contamination.

### 4. The Role of Synthetic Data Generation
Generating fine-tuning data using a stronger model (e.g., using GPT-4 to generate training data for a smaller model) has become standard practice. Stanford Alpaca used this approach to generate 52K examples from just 175 seed tasks. The key risks are: (a) the student model inherits the teacher's biases and errors, (b) model collapse from training on synthetic data over multiple generations, and (c) potential license restrictions (some model providers prohibit using outputs for training competitors). When using synthetic data, always validate a sample manually and mix in human-written examples.

### 5. Formatting Consistency
Pick one data format and stick with it throughout your dataset. Mixing Alpaca-style and chat-style formats in the same training run confuses the model. Ensure consistent use of special tokens, system prompts, and response formatting. If your model will use markdown in production, train it on markdown-formatted outputs.

### 6. Iterative Refinement
Fine-tuning data is never "done." The best datasets are built iteratively: train a model, evaluate it, identify failure modes, generate or collect data that addresses those failures, retrain, and repeat. This feedback loop is more effective than trying to build the perfect dataset upfront.

---

## Summary

| Format | Use Case | Key Fields | Example Method |
|--------|----------|------------|----------------|
| **Alpaca/SFT** | Supervised fine-tuning | `instruction`, `input`, `output` | Train on input-output pairs |
| **Chat** | Multi-turn conversations | `messages` (role + content) | Train on conversation flows |
| **RLHF Preference** | Reward model training | `prompt`, `chosen`, `rejected` | Train reward model, then PPO |
| **DPO** | Direct preference optimization | `prompt`, `chosen`, `rejected` | Directly optimize policy |

Understanding these formats is the first step toward building effective fine-tuning pipelines. The next notebook in this series will cover the actual training process and how to evaluate fine-tuned models.